In [2]:
import polars as pl

In [8]:
go_relations = catalog.load("bronze.ontology.go_relations")
go_terms = catalog.load("bronze.ontology.go_terms")

[06/24/25 20:24:13] INFO     Loading data from bronze.ontology.go_relations               ]8;id=788600;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=433263;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

                    INFO     Loading data from bronze.ontology.go_terms (CSVDataset)...   ]8;id=804171;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=337869;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\

In [10]:
go_terms

id,name,type
str,str,str
"""GO_0072278""","""metanephric comma-shaped body …","""biological_process"""
"""GO_0006312""","""mitotic recombination""","""biological_process"""
"""GO_0061449""","""olfactory bulb tufted cell dev…","""biological_process"""
"""GO_1903415""","""flavonoid transport from endop…","""biological_process"""
"""GO_1902657""","""protein localization to prospo…","""biological_process"""
…,…,…
"""GO_0018046""","""obsolete C-terminal peptidyl-m…","""biological_process"""
"""GO_0061854""","""positive regulation of neurobl…","""biological_process"""
"""GO_0072613""",null,"""CLASS"""


In [11]:
go_relations

tail,edge_type,head
str,str,str
"""GO_0000001""","""is_a""","""GO_0048308"""
"""GO_0000001""","""is_a""","""GO_0048311"""
"""GO_0000001""","""GOREL_0002003""","""GO_0005739"""
"""GO_0000002""","""is_a""","""GO_0007005"""
"""GO_0000006""","""is_a""","""GO_0005385"""
…,…,…
"""GO_2001316""","""is_a""","""GO_0120254"""
"""GO_2001317""","""is_a""","""GO_0034309"""
"""GO_2001317""","""is_a""","""GO_0042181"""


In [20]:
cc = go_terms.filter(pl.col("type") == "cellular_component")

df_go_edges = go_relations.rename({"tail": "x_id", "head": "y_id"})

(
    df_go_edges.join(
        cc.select(["id", "name"]), left_on="x_id", right_on="id", how="inner"
    )
    .rename({"name": "x_name"})
    .join(cc.select(["id", "name"]), left_on="y_id", right_on="id", how="inner")
    .rename({"name": "y_name"})
    .with_columns(
        [
            pl.lit("cellular_component").alias("x_type"),
            pl.lit("GO").alias("x_source"),
            pl.lit("cellular_component").alias("y_type"),
            pl.lit("GO").alias("y_source"),
            pl.lit("cellcomp_cellcomp").alias("relation"),
            pl.lit("parent-child").alias("relation_type"),
        ]
    )
    .with_columns(
        pl.concat_list([pl.col("x_id"), pl.col("y_id")]).list.sort().alias("sorted_ids")
    )
    .unique("sorted_ids", keep="first")
    .drop("sorted_ids")
    .select(
        [
            "relation",
            "relation_type",
            "x_id",
            "x_type",
            "x_name",
            "x_source",
            "y_id",
            "y_type",
            "y_name",
            "y_source",
        ]
    )
)

relation,relation_type,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
str,str,str,str,str,str,str,str,str,str
"""cellcomp_cellcomp""","""parent-child""","""GO_0061645""","""cellular_component""","""endocytic patch""","""GO""","""GO_0110165""","""cellular_component""","""cellular anatomical structure""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_0062073""","""cellular_component""","""histone mRNA stem-loop binding…","""GO""","""GO_0005737""","""cellular_component""","""cytoplasm""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_1990469""","""cellular_component""","""Rhino-Deadlock-Cutoff Complex""","""GO""","""GO_1990923""","""cellular_component""","""PET complex""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_0019823""","""cellular_component""","""P5 peroxisome""","""GO""","""GO_0005777""","""cellular_component""","""peroxisome""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_0061638""","""cellular_component""","""CENP-A containing chromatin""","""GO""","""GO_0034506""","""cellular_component""","""chromosome, centromeric core d…","""GO"""
…,…,…,…,…,…,…,…,…,…
"""cellcomp_cellcomp""","""parent-child""","""GO_0017117""","""cellular_component""","""single-stranded DNA-dependent …","""GO""","""GO_0033202""","""cellular_component""","""DNA helicase complex""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_0000939""","""cellular_component""","""inner kinetochore""","""GO""","""GO_0000776""","""cellular_component""","""kinetochore""","""GO"""
"""cellcomp_cellcomp""","""parent-child""","""GO_0019005""","""cellular_component""","""SCF ubiquitin ligase complex""","""GO""","""GO_0031461""","""cellular_component""","""cullin-RING ubiquitin ligase c…","""GO"""
